In [2]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm
import time

from scipy.signal import resample_poly

WINDOWING SAMPLERATE AND OVERLAP

In [3]:
FS = 200                     # Sampling frequency (Hz)
WINDOW_SEC = 30              # Sliding window length (seconds)
WINDOW_SAMPLES = FS * WINDOW_SEC
STEP_SEC = 1                 # Step size for 1 Hz output



    Resample signal using polyphase filtering

In [ ]:
def resample_signal(signal, fs_in=256, fs_out=200):

    return resample_poly(signal, fs_out, fs_in)

In [5]:
import warnings
warnings.filterwarnings("ignore")


FEATURE EXTRACTION VER 1

In [6]:
def extract_features_from_full_gsr_ppg(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # Resample signals to 200 Hz

    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Process signals
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(ppg, sampling_rate=FS)
    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    # -------------------------
    # RR intervals (IBI )
    # -------------------------
    rpeaks = info_ppg["PPG_Peaks"]
    rr_intervals = np.diff(rpeaks) / FS * 1000
    rr_times = rpeaks[1:]

    features = []
    step_samples = FS * STEP_SEC

    for start in range(0, len(df) - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # HR
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # HRV
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        if len(rr_win) >= 5:
            diff_rr = np.diff(rr_win)
            window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
            window["HRV_SDNN"] = np.std(rr_win, ddof=1)
        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

        # EDA
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()
        window["SCR_Count"] = int(eda_window["SCR_Peaks"].sum())

        # Metadata
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

FEATURE EXTRACTION WITH RR VALUES FOR ANALYS and SCR_RATE

In [19]:
def extract_features_with_rr(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # -------------------------
    # Resample signals to 200 Hz
    # -------------------------
    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Process signals
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(ppg, sampling_rate=FS)
    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    # -------------------------
    # RR intervals (IBI)
    # -------------------------
    rpeaks = info_ppg["PPG_Peaks"]

    rr_intervals = np.diff(rpeaks) / FS * 1000  # ms

    # bättre tidsrepresentation (mitten av intervallet)
    rr_times = (rpeaks[1:] + rpeaks[:-1]) / 2

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(ppg)  # 🔥 FIX: använd resamplad signal

    # -------------------------
    # Sliding windows
    # -------------------------
    for start in range(0, signal_len - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # -------------------------
        # HR
        # -------------------------
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # -------------------------
        # HRV + RR diagnostics
        # -------------------------
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        # filtrera orimliga RR
        rr_win = rr_win[(rr_win > 300) & (rr_win < 1500)]

        if len(rr_win) >= 5:
            diff_rr = np.diff(rr_win)

            # filtrera extrema diff
            diff_rr = diff_rr[np.abs(diff_rr) < 200]

            # HRV
            window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR diagnostics 
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_diff_std"] = np.std(diff_rr)

            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan

            window["RR_valid"] = 0

        # -------------------------
        # EDA
        # -------------------------
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()
       
        scr_count = int(eda_window["SCR_Peaks"].sum())

        window["SCR_Rate"] = scr_count/ WINDOW_SEC

        

        # -------------------------
        # Metadata
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

FEATURE EXTRACTION FOR TRAIN PART 1-48 FULL FILE (full_gsr_ppg.csv)

In [20]:
DATA_DIR = Path("../data/raw/Participants")

# -------------------------
# Collect participant files
# -------------------------
participant_files = sorted(
    DATA_DIR.rglob("full_gsr_ppg*.csv"),
    key=lambda x: int(x.parent.name.replace("Part", ""))
)
print("Found files:", len(participant_files))
for f in participant_files[:5]:
    print(f)
# -------------------------
# Select participants 1–48
# -------------------------
participant_files = participant_files[:48]

all_features = []

start_time = time.time()

for i, file in enumerate(tqdm(participant_files, desc="Processing participants")):
    participant_id = file.parent.name

    print(f"\nProcessing {participant_id} ({i+1}/{len(participant_files)})")

    df = pd.read_csv(file)

    participant_features = extract_features_with_rr(
        df, participant_id
    )

    all_features.append(participant_features)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    print(
        f"Elapsed: {elapsed/60:.1f} min | "
        f"Remaining: {remaining/60:.1f} min"
    )


Found files: 60
..\data\raw\Participants\Part1\full_gsr_ppg.csv
..\data\raw\Participants\Part2\full_gsr_ppg.csv
..\data\raw\Participants\Part3\full_gsr_ppg.csv
..\data\raw\Participants\Part4\full_gsr_ppg.csv
..\data\raw\Participants\Part5\full_gsr_ppg.csv


Processing participants:   0%|          | 0/48 [00:00<?, ?it/s]


Processing Part1 (1/48)


Processing participants:   2%|▏         | 1/48 [00:23<18:27, 23.55s/it]

Elapsed: 0.4 min | Remaining: 18.5 min

Processing Part2 (2/48)


...

Elapsed: 16.6 min | Remaining: 0.7 min

Processing Part47 (47/48)


Processing participants:  98%|█████████▊| 47/48 [16:52<00:17, 17.16s/it]

Elapsed: 16.9 min | Remaining: 0.4 min

Processing Part48 (48/48)


Processing participants: 100%|██████████| 48/48 [17:14<00:00, 21.56s/it]

Elapsed: 17.2 min | Remaining: 0.0 min


DF CHECK BEFORE LOG AND CLIPPING 

In [21]:
features_df = pd.concat(all_features, ignore_index=True)
features_df = features_df.dropna().reset_index(drop=True)
features_df.describe()


,HR,HRV_RMSSD,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_diff_std,RR_valid,EDA_Tonic,EDA_Phasic,SCR_Rate,time_sec
count,108428.000000,108428.000000,108428.000000,108428.000000,108428.000000,108428.000000,108428.000000,108428.000000,108428.000000,108428.0,108428.000000,108428.000000,108428.000000,108428.000000
mean,79.283488,54.870827,63.335001,772.415251,62.503440,633.108883,924.111254,39.567879,54.608549,1.0,2627.212835,-0.063104,0.028119,1132.651824
std,11.097128,22.455123,31.107675,106.814237,30.697477,129.364911,147.000136,5.555503,22.211801,0.0,11666.150181,80.416330,0.042426,657.187814
min,52.596467,6.833602,9.829522,424.857143,9.738082,305.000000,460.000000,26.000000,6.832951,1.0,-699.751593,-3659.799728,0.000000,0.000000
25%,71.396398,36.801615,41.726876,698.372093,41.199121,560.000000,830.000000,36.000000,36.740709,1.0,84.713174,-0.125804,0.000000,564.000000
50%,78.343750,53.488340,56.681599,767.051282,55.946375,645.000000,915.000000,39.000000,53.308797,1.0,171.337658,-0.000463,0.000000,1129.000000
75%,86.078315,71.789239,77.275378,841.027778,76.199321,715.000000,1010.000000,43.000000,71.413369,1.0,368.120550,0.124713,0.033333,1699.000000
max,141.215822,133.037076,242.791954,1142.115385,239.737845,1005.000000,1495.000000,71.000000,132.993390,1.0,92911.198628,4880.031171,0.400000,2309.000000


LOG AND CLIPPING FEATURES 

In [22]:

# EDA stabilization, log-transform and clip extreme values

EPS = 1e-6

features_df["EDA_Tonic_log"] = np.log(features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS)
features_df["EDA_Phasic_log"] = np.log(np.abs(features_df["EDA_Phasic"]) + EPS)

# SCR LOG
features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])

#Clip extreme values

features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0,200)
features_df["HRV_SDNN"] = features_df["HRV_SDNN"].clip(0,200)


# EDA CLIPP
features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

# SCR CLIPP

features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)


SAVE FULL FILE FOR ANALYS

In [23]:
features_df.to_csv("../data/features_extraction_full_file_analys_SCRRate.csv", index=False)

CREATE DF FOR SOM MODEL WITH CORRECT FEATURES

In [27]:
# Feature columns used for SOM
FEATURE_COLS = [
    "HR",
    "HRV_RMSSD",
    "HRV_SDNN",
    "SCR_Rate",
    "EDA_Tonic_log",
    "EDA_Phasic_log"
]

# SOM input matrix
X_features = features_df[FEATURE_COLS]

# (Optional) metadata kept separately
metadata = features_df[["participant", "time_sec"]]

X_features.describe()

,HR,HRV_RMSSD,HRV_SDNN,SCR_Rate,EDA_Tonic_log,EDA_Phasic_log
count,108428.000000,108428.000000,108428.000000,108428.000000,108428.000000,108428.000000
mean,79.283488,54.870827,63.313658,0.026782,6.929523,-1.942847
std,11.097128,22.455123,31.007067,0.039017,0.412408,1.545011
min,52.596467,6.833602,9.829522,0.000000,6.000000,-4.000000
25%,71.396398,36.801615,41.726876,0.000000,6.665002,-3.138942
50%,78.343750,53.488340,56.681599,0.000000,6.769744,-2.077487
75%,86.078315,71.789239,77.275378,0.032790,6.973423,-1.054910
max,141.215822,133.037076,200.000000,0.200000,8.000000,2.000000


In [29]:
print(features_df["SCR_Rate"].describe())

count    108428.000000
mean          0.026782
std           0.039017
min           0.000000
25%           0.000000
50%           0.000000
75%           0.032790
max           0.200000
Name: SCR_Rate, dtype: float64


SAVE FULL FILE FOR SOM MODEL

In [30]:
X_features.to_csv(    
    "../data/features_full_file_SOM_ScrRate.csv",
    index=False
)


FEATURE EXTRACTION 49-60 FOR TEST with clipping and log

FULL FILE (full_gsr_ppg.csv)

SAVED IN /test_features_windows --> 1: all participants 2: participant_49..60

In [31]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import os

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR = Path("../data/raw/Participants")
SAVE_DIR = Path("test_features_windows")
os.makedirs(SAVE_DIR, exist_ok=True)

EPS = 1e-6

# -------------------------
# Collect participant files
# -------------------------
participant_files = sorted(
    DATA_DIR.rglob("full_gsr_ppg*.csv"),
    key=lambda x: int(x.parent.name.replace("Part", ""))
)

print("Total files found:", len(participant_files))

# -------------------------
# Select participants 49–60
# -------------------------
participant_files = participant_files[48:60]

print("Selected participants:")
for f in participant_files:
    print(f.parent.name)

# -------------------------
# PROCESSING
# -------------------------
all_features = []
start_time = time.time()

for i, file in enumerate(tqdm(participant_files, desc="Processing participants")):

    participant_id = int(file.parent.name.replace("Part", ""))
    tqdm.write(f"Processing Part{participant_id} ({i+1}/{len(participant_files)})")

    df = pd.read_csv(file)

    # -------------------------
    # FEATURE EXTRACTION WITH RR
    # -------------------------
    features_df = extract_features_with_rr(df, participant_id)

    # -------------------------
    # EDA STABILIZATION 
    # -------------------------
    features_df["EDA_Tonic_log"] = np.log(
        features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS
    )

    features_df["EDA_Phasic_log"] = np.log(
        np.abs(features_df["EDA_Phasic"]) + EPS
    )

    # -------------------------
    # CLIPPING 
    # -------------------------
    features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0, 200)
    features_df["HRV_SDNN"]  = features_df["HRV_SDNN"].clip(0, 200)

   
    # SCR LOG
    features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])
    features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)

    

    # EDA clipping 
    features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
    features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

    # -------------------------
    # DROP RAW EDA 
    # -------------------------
    features_df = features_df.drop(columns=["EDA_Tonic", "EDA_Phasic"])

    # -------------------------
    # SAVE PER PARTICIPANT
    # -------------------------
    features_df.to_csv(
        SAVE_DIR / f"participant_{participant_id}_windows_features_new_SCR.csv",
        index=False
    )

    all_features.append(features_df)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    tqdm.write(
        f"Elapsed: {elapsed/60:.1f} min | Remaining: {remaining/60:.1f} min"
    )

# -------------------------
# COMBINE ALL
# -------------------------
features_all_df = pd.concat(all_features, ignore_index=True)

features_all_df.to_csv(
    SAVE_DIR / "features_49_60_ALL_WINDOWS_features_new_SCR.csv",
    index=False
)

print("\n✅ DONE")
print(features_all_df.shape)

Total files found: 60
Selected participants:
Part49
Part50
Part51
Part52
Part53
Part54
Part55
Part56
Part57
Part58
Part59
Part60


Processing participants:   0%|          | 0/12 [00:00<?, ?it/s]

Processing Part49 (1/12)


Processing participants:   8%|▊         | 1/12 [00:26<04:52, 26.55s/it]

Elapsed: 0.4 min | Remaining: 4.9 min
Processing Part50 (2/12)


...

Elapsed: 4.0 min | Remaining: 0.8 min
Processing Part59 (11/12)


Processing participants:  92%|█████████▏| 11/12 [04:27<00:24, 24.94s/it]

Elapsed: 4.5 min | Remaining: 0.4 min
Processing Part60 (12/12)


Processing participants: 100%|██████████| 12/12 [04:46<00:00, 23.84s/it]


Elapsed: 4.8 min | Remaining: 0.0 min

✅ DONE
(27545, 15)


FEATURE EXTRACTION BY_BLOCK 

FILE PATH -->  file = p_dir / "by_block" / "7_gsr_ppg_.csv" (CHANGE FOR DIFFERENT BLOCK)

SAVE BOTH ALL PARTICIPANT AND ONE PER PARTICIPANT 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import os

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR = Path("../data/raw/Participants")
SAVE_DIR = Path("test_features_calm_7")
os.makedirs(SAVE_DIR, exist_ok=True)

EPS = 1e-6

# -------------------------
# Collect stroop files
# -------------------------
participant_dirs = sorted(
    [p for p in DATA_DIR.glob("Part*")],
    key=lambda x: int(x.name.replace("Part", ""))
)

# Select 49–60
participant_dirs = participant_dirs[48:60]

print("Selected participants:")
for p in participant_dirs:
    print(p.name)

# -------------------------
# PROCESSING
# -------------------------
all_features = []
start_time = time.time()

for i, p_dir in enumerate(tqdm(participant_dirs, desc="Processing participants")):

    participant_id = int(p_dir.name.replace("Part", ""))
    tqdm.write(f"Processing Part{participant_id} ({i+1}/{len(participant_dirs)})")

    # -------------------------
    # FILE PATH (stroop)
    # -------------------------
    file = p_dir / "by_block" / "7_gsr_ppg_.csv"

    if not file.exists():
        tqdm.write(f"⚠️ Missing file for Part{participant_id}")
        continue

    df = pd.read_csv(file)

    # -------------------------
    # FEATURE EXTRACTION
    # -------------------------
    features_df = extract_features_from_full_gsr_ppg(df, participant_id)

    # -------------------------
    # EDA STABILIZATION
    # -------------------------
    features_df["EDA_Tonic_log"] = np.log(
        features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS
    )

    features_df["EDA_Phasic_log"] = np.log(
        np.abs(features_df["EDA_Phasic"]) + EPS
    )

    # -------------------------
    # CLIPPING
    # -------------------------
    features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0, 200)
    features_df["HRV_SDNN"]  = features_df["HRV_SDNN"].clip(0, 200)

    features_df["SCR_Count"] = np.log1p(features_df["SCR_Count"])

    features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
    features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

    # -------------------------
    # DROP RAW EDA
    # -------------------------
    features_df = features_df.drop(columns=["EDA_Tonic", "EDA_Phasic"])

    # -------------------------
    # SAVE PER PARTICIPANT
    # -------------------------
    features_df.to_csv(
        SAVE_DIR / f"participant_{participant_id}_calm_windows7.csv",
        index=False
    )

    all_features.append(features_df)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_dirs) - (i + 1))

    tqdm.write(
        f"Elapsed: {elapsed/60:.1f} min | Remaining: {remaining/60:.1f} min"
    )

# -------------------------
# COMBINE ALL
# -------------------------
features_all_df = pd.concat(all_features, ignore_index=True)

features_all_df.to_csv(
    SAVE_DIR / "stroop_49_60_ALL_WINDOWS.csv",
    index=False
)

print("\n✅ DONE")
print(features_all_df.shape)